# Phase 10: Fold Robustness Validation

## 1. Packages and paths

In [7]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from itertools import product
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

warnings.filterwarnings('default')
np.random.seed(36)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths
PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models_folds'
PHASE10_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase10_fold_robustness'
PHASE10_OUTPUT.mkdir(parents=True, exist_ok=True)

#Constants
DATASET='2007'
FOLDS=['Fold1', 'Fold2', 'Fold3']
MODELS=['pointwise', 'pairwise', 'lightgbm']
PIPELINES=['raw', 'global', 'per_query']
EXPECTED_CONFIGS=len(MODELS) * len(PIPELINES)  # 9

BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'
MIN_SAMPLE_WARNING=10

warnings_issued=[]

print('='*80)
print('PHASE 10: FOLD ROBUSTNESS VALIDATION')
print('='*80)
print(f'Output: {PHASE10_OUTPUT}')
print(f'Dataset: MQ{DATASET}')
print(f'Folds: {FOLDS}')
print(f'Expected configs/fold: {EXPECTED_CONFIGS}')
print(f'Baseline: {BASELINE_MODEL}_{BASELINE_PIPELINE}')
print('='*80)

PHASE 10: FOLD ROBUSTNESS VALIDATION
Output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase10_fold_robustness
Dataset: MQ2007
Folds: ['Fold1', 'Fold2', 'Fold3']
Expected configs/fold: 9
Baseline: pointwise_raw


## 2. Util functions

In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    make_fail_flag,
    check_sample_size,
    safe_k,
    compute_failure_at_k,
    mcnemar_test,
    bh_fdr,
)

In [9]:
def compute_score_gap(pred_df: pd.DataFrame, qids: set, k: int=5, relevance_threshold: int=1) -> dict:
    """
    Score gap=(best relevant score)-(score at rank k).
    Returns {qid: gap or np.nan}.
    """
    gaps={}
    for qid in qids:
        q_docs=pred_df[pred_df["qid"] == qid].copy()
        if len(q_docs)==0:
            gaps[qid]=np.nan
            continue

        q_docs=q_docs.sort_values("score", ascending=False)

        relevant=q_docs[q_docs["label"]>=relevance_threshold]
        if len(relevant)==0:
            gaps[qid]=np.nan
            continue

        best_rel=float(relevant["score"].max())

        k_actual=min(k, len(q_docs))
        if k_actual<=0:
            gaps[qid]=np.nan
            continue

        score_at_k=float(q_docs.iloc[k_actual-1]["score"])
        gaps[qid]=best_rel-score_at_k

    return gaps

print("imported src.utils + defined compute_score_gap")

imported src.utils + defined compute_score_gap


## 3. Loading Phase 6 Artifacts

In [10]:
print('\n'+'='*80)
print('LOADING PHASE 6 ARTIFACTS')
print('='*80)

all_fold_artifacts={}

for fold in FOLDS:
    print(f'\n{fold}:')
    artifacts={'query_metrics': {}, 'predictions': {}}
    
    for model, pipeline in product(MODELS, PIPELINES):
        key=f'{model}_{pipeline}_{DATASET}'
        qm_file=PHASE6_OUTPUT / fold/f'{key}_query_metrics.csv'
        pred_file=PHASE6_OUTPUT / fold/f'{key}_predictions.csv'
        
        if not qm_file.exists() or not pred_file.exists():
            print(f'Missing: {key}')
            continue
        
        qm=pd.read_csv(qm_file)
        pred=pd.read_csv(pred_file)
        
        #Proper validation with flag
        ok=True
        for col in ['qid', 'num_docs', 'num_relevant_1', 'Failure@5_primary']:
            if col not in qm.columns:
                print(f'{key} missing column: {col}')
                ok=False
        for col in ['qid', 'label', 'score']:
            if col not in pred.columns:
                print(f'{key} missing column: {col}')
                ok=False
        if not ok:
            continue
        
        #Coercing dtypes
        qm['qid']=qm['qid'].astype(int)
        pred['qid']=pred['qid'].astype(int)
        pred['label']=pred['label'].astype(int)
        pred['score']=pred['score'].astype(float)
        
        artifacts['query_metrics'][key]=qm
        artifacts['predictions'][key]=pred
    
    if len(artifacts['query_metrics'])>0:
        all_fold_artifacts[fold]=artifacts
        print(f'{len(artifacts["query_metrics"])} configs')
    else:
        msg=f'{fold}: no configs loaded'
        warnings_issued.append(msg)
        print(f'{msg}')

print(f'\nLoaded {len(all_fold_artifacts)}/{len(FOLDS)} folds')
print('='*80)


LOADING PHASE 6 ARTIFACTS

Fold1:
9 configs

Fold2:
9 configs

Fold3:
9 configs

Loaded 3/3 folds
